# 17. 강화학습 실습 — Q-Learning부터 DQN까지

> **제17장** · **이론편 대응: 14장 (강화학습)**
> **예상 소요**: 70분
> **필요 사양**: **[CPU]** 로 실행 가능
> **추가 설치**: **Gymnasium** (1절 참조)
> **다운로드**: 없음 (환경이 코드로 생성됨)

---

## 이 장에서 하는 일

지금까지는 정답이 있는 데이터로 학습했다. 강화학습은 **정답 없이 보상만 주어지는** 상황을 다룬다.

| 절 | 하는 일 | 이론편 대응 |
|---|---|---|
| 1 | **Gymnasium 설치** | — |
| 1 | 환경과 상호작용 | 14.1절 |
| 2 | **Q-Learning 손계산 검증 (0→5.0→2.25)** ★ | 14.3절 |
| 3 | 표 기반 Q-Learning — FrozenLake | 14.3절 |
| 4 | 탐험과 활용 | 14.4절 |
| 5 | DQN — 표가 불가능할 때 | 14.5절 |
| 6 | 학습 과정 분석 | 14.5절 |

---

## 1. 준비 — Gymnasium 설치

이 장부터 새 패키지가 필요하다.

### Gymnasium이란

강화학습 실험용 **환경 모음**이다. 게임·제어 문제를 표준화된 방식으로 제공한다.
원래 OpenAI가 만든 Gym을 Farama Foundation이 이어받아 관리하고 있다.

| 항목 | 내용 |
|---|---|
| 패키지 이름 | `gymnasium` (예전 이름 `gym`이 아니다) |
| 관리 주체 | Farama Foundation |
| 문서 | `https://gymnasium.farama.org` |
| 용량 | 약 1MB (기본 환경만) |

### 설치 명령

터미널에서 (가상환경 활성화 상태로) 실행한다.

```
pip install gymnasium
```

**주의**: 예전 자료에는 `pip install gym`이라고 되어 있는 경우가 많다.
`gym`은 더 이상 갱신되지 않으므로 `gymnasium`을 써야 한다.
API도 조금 달라졌는데, 1절에서 그 차이를 다룬다.

### 아타리 게임 등 추가 환경이 필요하다면

이 장은 기본 설치만으로 충분하다. 나중에 더 복잡한 환경을 쓰고 싶다면 다음을 참고한다.

```
pip install "gymnasium[classic-control]"   # 이 장에서 쓰는 것 (기본 포함)
pip install "gymnasium[box2d]"             # LunarLander 등 (컴파일러 필요할 수 있음)
pip install "gymnasium[atari]"             # 아타리 게임
```

아래 셀로 설치 상태를 확인한다.

In [ ]:
import importlib
import sys

print("=" * 55)
print("필요 패키지 확인")
print("=" * 55)

required = [
    ("gymnasium", "강화학습 환경", "pip install gymnasium"),
    ("torch",     "신경망 (6절 DQN)", "pip install torch --index-url https://download.pytorch.org/whl/cu128"),
    ("numpy",     "수치 계산", "pip install numpy"),
    ("matplotlib","시각화", "pip install matplotlib"),
]

missing = []
for name, desc, install in required:
    try:
        mod = importlib.import_module(name)
        ver = getattr(mod, "__version__", "?")
        print(f"[OK]   {name:<14}{ver:<12}{desc}")
    except ImportError:
        print(f"[없음] {name:<14}{'':<12}{desc}")
        missing.append((name, install))

print("-" * 55)
if missing:
    print("설치가 필요합니다. 터미널에서 실행하세요:")
    for name, cmd in missing:
        print(f"  {cmd}")
    print()
    print("설치 후 이 장의 커널을 재시작해야 반영됩니다.")
    print("  (VS Code 상단의 'Restart' 버튼)")
else:
    print("[준비 완료] 2절로 진행하세요.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform

_c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
      "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
_a = {f.name for f in fm.fontManager.ttflist}
for _n in _c.get(platform.system(), []):
    if _n in _a:
        plt.rcParams["font.family"] = _n
        break
plt.rcParams["axes.unicode_minus"] = False

try:
    import gymnasium as gym
    print(f"gymnasium {gym.__version__}")
    HAS_GYM = True
except ImportError:
    HAS_GYM = False
    print("gymnasium 미설치 — 1절을 참조하세요")
    print("(3절의 Q-Learning 손계산 검증은 gymnasium 없이도 실행됩니다)")

---

## 2. 환경과 상호작용 — 이론편 14.1절

이론편 14.1절에서 다룬 강화학습의 구성 요소를 코드로 확인한다.

| 요소 | 뜻 | Gymnasium |
|---|---|---|
| 환경(Environment) | 에이전트가 놓인 세계 | `gym.make(...)` |
| 상태(State) | 지금 상황 | `obs` |
| 행동(Action) | 할 수 있는 일 | `env.step(action)` |
| 보상(Reward) | 행동의 결과로 받는 점수 | `reward` |

**지도학습과의 차이**는 "정답이 없다"는 점이다. 무엇이 옳은 행동인지 아무도 알려주지 않고,
행동한 뒤에 보상만 돌아온다.

In [ ]:
import gymnasium as gym
import numpy as np

# CartPole: 막대가 쓰러지지 않게 수레를 좌우로 움직이는 문제
env = gym.make("CartPole-v1")

print("=" * 60)
print("CartPole-v1 환경")
print("=" * 60)
print(f"관측 공간 : {env.observation_space}")
print(f"  → 4개 값: 수레 위치, 수레 속도, 막대 각도, 막대 각속도")
print()
print(f"행동 공간 : {env.action_space}")
print(f"  → 2가지: 0=왼쪽으로 밀기, 1=오른쪽으로 밀기")
print()

# 환경 초기화
obs, info = env.reset(seed=42)
print(f"초기 상태: {obs.round(4)}")
print()

# 한 걸음 진행
action = 0
obs, reward, terminated, truncated, info = env.step(action)

print(f"행동 {action}(왼쪽)을 취한 결과")
print(f"  새 상태  : {obs.round(4)}")
print(f"  보상     : {reward}")
print(f"  종료됨   : {terminated}   ← 막대가 쓰러졌는가")
print(f"  잘렸음   : {truncated}    ← 시간 제한에 걸렸는가")
env.close()

print()
print("보상이 매 스텝 1점이다. 오래 버틸수록 총점이 높아진다.")
print("최대 500점 (500스텝 버티면 truncated=True)")

### Gym과 Gymnasium의 API 차이

예전 자료를 보다 보면 오류가 나는 경우가 있다. 반환값 개수가 달라졌기 때문이다.

| | 예전 `gym` | 현재 `gymnasium` |
|---|---|---|
| `reset()` | `obs` | `obs, info` |
| `step()` | `obs, reward, done, info` (4개) | `obs, reward, terminated, truncated, info` (5개) |

`done`이 두 개로 나뉜 것이 핵심이다.

- `terminated`: 실제로 끝났다 (막대가 쓰러짐, 목표 도달)
- `truncated`: 시간 제한으로 잘렸다 (아직 실패한 것은 아님)

이 구분이 중요한 이유는 **가치 계산이 달라지기 때문**이다. 시간 제한으로 잘린 것은
"거기서 끝"이 아니므로 미래 가치를 0으로 두면 안 된다.

In [ ]:
import gymnasium as gym
import numpy as np

# 무작위로 행동하면 얼마나 버티는지 확인
env = gym.make("CartPole-v1")
rng = np.random.RandomState(0)

print("=" * 55)
print("무작위 행동 (아무것도 학습하지 않은 상태)")
print("=" * 55)

scores = []
for episode in range(10):
    obs, info = env.reset(seed=episode)
    total = 0
    for step in range(500):
        action = env.action_space.sample()     # 무작위 선택
        obs, reward, terminated, truncated, info = env.step(action)
        total += reward
        if terminated or truncated:
            break
    scores.append(total)
    print(f"  에피소드 {episode+1:2}: {total:5.0f}점  ({int(total)}스텝 버팀)")

env.close()
print("-" * 55)
print(f"평균 {np.mean(scores):.1f}점 / 최대 500점")
print()
print("무작위로는 20~30스텝밖에 못 버틴다.")
print("6절에서 학습을 통해 이를 크게 개선한다.")

---

## 3. Q-Learning 손계산 검증 — 이론편 14.3절 ★

이론편 14.3절에서 두 상태(A, B)만 있는 예로 Q값 갱신을 손으로 계산했다.

**갱신 규칙**

$$Q(s,a) \leftarrow Q(s,a) + \alpha\Big[\underbrace{r + \gamma\max_{a'}Q(s',a')}_{\text{목표값}} - \underbrace{Q(s,a)}_{\text{현재 추정}}\Big]$$

**이론편의 설정과 결과**

| 회차 | 상황 | Q값 변화 |
|---|---|---|
| 1 | A → B (보상 0) | Q(A): 0 → **0** |
| 2 | B → 목표 (보상 10) | Q(B): 0 → **5.0** |
| 3 | A → B (보상 0) | Q(A): 0 → **2.25** |

**3회차가 핵심이다.** A에서는 보상을 한 번도 받은 적이 없는데 값이 생겼다.

In [ ]:
import numpy as np

# 이론편 14.3절과 완전히 같은 설정
ALPHA = 0.5     # 학습률
GAMMA = 0.9     # 할인율

Q = {"A": 0.0, "B": 0.0}

# (현재 상태, 받은 보상, 다음 상태) — 다음 상태가 None이면 종료
episodes = [
    ("A", 0,  "B"),      # 1회차
    ("B", 10, None),     # 2회차 — 여기서만 보상을 받는다
    ("A", 0,  "B"),      # 3회차
]

print("=" * 68)
print("이론편 14.3절 Q-Learning 손계산 검증")
print("=" * 68)
print(f"학습률 alpha={ALPHA}, 할인율 gamma={GAMMA}")
print()

for i, (s, r, s_next) in enumerate(episodes, 1):
    max_q_next = 0.0 if s_next is None else Q[s_next]
    old = Q[s]

    target = r + GAMMA * max_q_next          # 목표값
    td_error = target - old                  # 시간차 오차
    Q[s] = old + ALPHA * td_error

    print(f"[{i}회차] 상태 {s}, 보상 {r}, 다음 {s_next if s_next else '종료'}")
    print(f"  목표값   = {r} + {GAMMA} x {max_q_next:.2f} = {target:.2f}")
    print(f"  TD 오차  = {target:.2f} - {old:.2f} = {td_error:.2f}")
    print(f"  Q({s})    = {old:.2f} + {ALPHA} x {td_error:.2f} = {Q[s]:.2f}")
    print()

print("-" * 68)
print(f"{'항목':<12}{'계산 결과':<16}{'이론편 값'}")
print("-" * 68)
print(f"{'Q(A)':<12}{Q['A']:<16.2f}2.25")
print(f"{'Q(B)':<12}{Q['B']:<16.2f}5.00")
print("-" * 68)

assert abs(Q["A"] - 2.25) < 1e-9
assert abs(Q["B"] - 5.00) < 1e-9
print("[OK] 이론편 14.3절 손계산과 일치")

### 이 결과가 뜻하는 것

3회차에서 **A는 보상을 받은 적이 없는데** Q값이 2.25가 되었다.

$$Q(A) = 0.5 \times \big[0 + 0.9 \times \underbrace{5.0}_{Q(B)}\big] = 2.25$$

B가 좋은 상태라는 사실이 알려지자, **B로 갈 수 있는 A도 덩달아 가치를 얻은 것**이다.

이것이 강화학습의 핵심 메커니즘이다. 보상은 목표 지점에서만 주어지지만,
반복을 거치며 그 가치가 앞쪽 상태로 한 칸씩 거슬러 전파된다.

바둑에서 마지막에 승패만 알려줘도 중반의 좋은 수를 배울 수 있는 것이 이 원리다.
다만 **전파에 시간이 걸리므로**, 목표까지 거리가 멀수록 학습에 필요한 반복이 크게 는다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 전파가 얼마나 걸리는지 확인 — 일자형 통로
N_STATES = 8      # 상태 0~7, 7이 목표
ALPHA, GAMMA = 0.5, 0.9

Q = np.zeros(N_STATES)
snapshots = []

for episode in range(30):
    s = 0
    while s < N_STATES - 1:
        s_next = s + 1
        r = 10.0 if s_next == N_STATES - 1 else 0.0
        max_next = 0.0 if s_next == N_STATES - 1 else Q[s_next]
        Q[s] += ALPHA * (r + GAMMA * max_next - Q[s])
        s = s_next
    if episode in (0, 1, 2, 5, 10, 29):
        snapshots.append((episode + 1, Q.copy()))

print("=" * 60)
print("가치가 앞으로 전파되는 과정 (8칸 통로, 끝에서만 보상 10)")
print("=" * 60)
print(f"{'에피소드':<10}" + "".join(f"{f'상태{i}':<9}" for i in range(N_STATES - 1)))
print("-" * 60)
for ep, q in snapshots:
    print(f"{ep:<10}" + "".join(f"{v:<9.3f}" for v in q[:-1]))
print("-" * 60)
print()
print("1회차에는 목표 바로 앞(상태6)만 값이 생긴다.")
print("회를 거듭하며 값이 뒤에서 앞으로 한 칸씩 번져 나간다.")

fig, ax = plt.subplots(figsize=(8, 4.5))
for ep, q in snapshots:
    ax.plot(range(N_STATES - 1), q[:-1], marker="o", label=f"{ep}회차")
ax.set_xlabel("상태 (7이 목표)")
ax.set_ylabel("Q값")
ax.set_title("가치의 전파")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---

## 4. 표 기반 Q-Learning — FrozenLake

이제 실제 환경에 적용한다. FrozenLake는 4×4 격자에서 구멍을 피해 목표에 도달하는 문제다.

| 기호 | 뜻 |
|---|---|
| S | 시작 |
| F | 얼음 (안전) |
| H | 구멍 (빠지면 끝) |
| G | 목표 (보상 1) |

상태가 16개, 행동이 4개뿐이므로 **Q값을 16×4 표에 전부 저장**할 수 있다.
이론편 14.3절에서 다룬 표 기반 방식이다.

In [ ]:
import gymnasium as gym
import numpy as np

# is_slippery=False: 미끄러지지 않게 (결정적 환경)
env = gym.make("FrozenLake-v1", is_slippery=False)

n_states = env.observation_space.n
n_actions = env.action_space.n

print("=" * 55)
print("FrozenLake-v1")
print("=" * 55)
print(f"상태 수 : {n_states}   (4x4 격자)")
print(f"행동 수 : {n_actions}   (0=왼, 1=아래, 2=오른, 3=위)")
print(f"Q 테이블 크기: {n_states} x {n_actions} = {n_states*n_actions}칸")
print()

print("지도")
desc = env.unwrapped.desc
for row in desc:
    print("  " + " ".join(c.decode() for c in row))
print()
print("  S=시작  F=얼음  H=구멍  G=목표")
print()
print("보상은 목표(G)에 도달했을 때만 1점, 나머지는 0점이다.")
print("→ 2절에서 본 '보상이 뒤에서 앞으로 전파되는' 상황")
env.close()

In [ ]:
import gymnasium as gym
import numpy as np


def train_q_learning(episodes=2000, alpha=0.8, gamma=0.95, seed=0):
    # 표 기반 Q-Learning (이론편 14.3절)
    env = gym.make("FrozenLake-v1", is_slippery=False)
    rng = np.random.RandomState(seed)
    Q = np.zeros((env.observation_space.n, env.action_space.n))

    rewards, epsilons = [], []

    for ep in range(episodes):
        s, _ = env.reset(seed=int(rng.randint(1_000_000)))
        # 탐험 비율을 서서히 줄인다 (4절에서 자세히)
        eps = max(0.05, 1.0 - ep / (episodes * 0.5))
        total = 0.0

        for _ in range(100):
            if rng.rand() < eps:
                a = env.action_space.sample()      # 탐험
            else:
                a = int(Q[s].argmax())             # 활용

            s2, r, term, trunc, _ = env.step(a)

            # 이론편 14.3절의 갱신 규칙
            max_next = 0.0 if term else Q[s2].max()
            Q[s, a] += alpha * (r + gamma * max_next - Q[s, a])

            s = s2
            total += r
            if term or trunc:
                break

        rewards.append(total)
        epsilons.append(eps)

    env.close()
    return Q, rewards, epsilons


print("=" * 55)
print("학습 중...")
Q_table, rewards, epsilons = train_q_learning()

print("=" * 55)
print("학습 결과")
print("=" * 55)
window = 100
for start in range(0, 2000, 400):
    rate = np.mean(rewards[start:start + window])
    print(f"  {start:4}~{start+window:4} 에피소드: 성공률 {rate:.2f}")
print()

# 학습된 정책으로 평가
env = gym.make("FrozenLake-v1", is_slippery=False)
wins = 0
for _ in range(100):
    s, _ = env.reset()
    for _ in range(100):
        s, r, term, trunc, _ = env.step(int(Q_table[s].argmax()))
        if term or trunc:
            wins += r
            break
env.close()

print(f"학습 후 성공률: {wins:.0f}/100")
if wins >= 90:
    print("[OK] 목표에 안정적으로 도달한다")
else:
    print("[주의] 성공률이 낮습니다. 아래를 시도해 보세요:")
    print("  - episodes 를 3000~5000 으로 늘리기")
    print("  - 이 셀을 다시 실행 (탐험이 어디로 향했느냐에 따라 결과가 달라짐)")
    print("  이런 불안정성 자체가 7절에서 다룰 강화학습의 특징이다.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 학습된 Q 테이블 들여다보기
arrows = ["←", "↓", "→", "↑"]

print("=" * 55)
print("학습된 정책 (각 칸에서 선택할 행동)")
print("=" * 55)

desc = [["S","F","F","F"],["F","H","F","H"],["F","F","F","H"],["H","F","F","G"]]
for i in range(4):
    row = []
    for j in range(4):
        s = i * 4 + j
        cell = desc[i][j]
        if cell in ("H", "G"):
            row.append(f" {cell} ")
        else:
            row.append(f" {arrows[int(Q_table[s].argmax())]} ")
    print("  " + "|".join(row))

print()
print("=" * 55)
print("Q값 (각 칸의 최대 Q)")
print("=" * 55)
for i in range(4):
    print("  " + " ".join(f"{Q_table[i*4+j].max():6.3f}" for j in range(4)))
print()
print("목표에 가까울수록 Q값이 크다 — 2절에서 본 전파의 결과다.")

fig, ax = plt.subplots(figsize=(6, 5))
values = Q_table.max(axis=1).reshape(4, 4)
im = ax.imshow(values, cmap="YlOrRd")
plt.colorbar(im, ax=ax, label="최대 Q값")
for i in range(4):
    for j in range(4):
        s = i * 4 + j
        cell = desc[i][j]
        label = cell if cell in ("H", "G") else arrows[int(Q_table[s].argmax())]
        ax.text(j, i, f"{label}\n{values[i,j]:.2f}", ha="center", va="center", fontsize=10)
ax.set_title("학습된 가치와 정책")
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()

---

## 5. 탐험과 활용 — 이론편 14.4절

위 코드에 `eps`라는 값이 있었다. 이론편 14.4절에서 다룬 **탐험(exploration)과 활용(exploitation)의 균형**이다.

| 전략 | 문제 |
|---|---|
| 항상 최선의 행동만 (활용) | 처음 찾은 길이 최선인지 모른다 |
| 항상 무작위 (탐험) | 배운 것을 쓰지 못한다 |

**ε-greedy**는 확률 ε로 무작위, 나머지는 최선을 고른다.
학습 초기에는 ε을 크게(많이 탐험), 나중에는 작게(배운 것 활용) 줄여 간다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import gymnasium as gym

print("=" * 60)
print("탐험 전략 비교")
print("=" * 60)

strategies = {}
for name, eps_fn in [
    ("항상 탐험 (eps=1.0)", lambda ep, n: 1.0),
    ("탐험 거의 없음 (eps=0.05)", lambda ep, n: 0.05),
    ("점차 줄임 (1.0 → 0.05)", lambda ep, n: max(0.05, 1.0 - ep / (n * 0.5))),
]:
    env = gym.make("FrozenLake-v1", is_slippery=False)
    rng = np.random.RandomState(0)
    Q = np.zeros((16, 4))
    rews = []
    N = 2000

    for ep in range(N):
        s, _ = env.reset(seed=int(rng.randint(1_000_000)))
        eps = eps_fn(ep, N)
        total = 0.0
        for _ in range(100):
            a = env.action_space.sample() if rng.rand() < eps else int(Q[s].argmax())
            s2, r, term, trunc, _ = env.step(a)
            Q[s, a] += 0.8 * (r + 0.95 * (0.0 if term else Q[s2].max()) - Q[s, a])
            s = s2; total += r
            if term or trunc: break
        rews.append(total)
    env.close()

    # 학습된 정책 평가
    env = gym.make("FrozenLake-v1", is_slippery=False)
    wins = 0
    for _ in range(100):
        s, _ = env.reset()
        for _ in range(100):
            s, r, term, trunc, _ = env.step(int(Q[s].argmax()))
            if term or trunc:
                wins += r; break
    env.close()

    strategies[name] = (rews, wins)
    print(f"  {name:<28} 최종 정책 성공률 {wins:.0f}/100")

print("-" * 60)
print()
print("항상 탐험하면 학습은 되지만(정책은 좋음) 학습 중 성적이 나쁘다.")
print("탐험이 너무 적으면 좋은 길을 못 찾을 수 있다.")

fig, ax = plt.subplots(figsize=(9, 4.5))
for name, (rews, _) in strategies.items():
    smooth = np.convolve(rews, np.ones(100)/100, mode="valid")
    ax.plot(smooth, label=name, linewidth=2)
ax.set_xlabel("에피소드")
ax.set_ylabel("성공률 (100회 이동평균)")
ax.set_title("탐험 전략에 따른 학습 곡선")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---

## 6. DQN — 표가 불가능할 때 (이론편 14.5절)

FrozenLake는 상태가 16개뿐이라 표에 다 담겼다. 하지만 CartPole은 다르다.

| 환경 | 상태 | 표 저장 |
|---|---|---|
| FrozenLake | 16개 (이산) | 가능 |
| **CartPole** | **4개 실수값 (연속)** | **불가능** |

막대 각도가 0.1도든 0.1001도든 각각 다른 상태다. **무한하다.**

이론편 14.5절에서 다룬 DQN의 해법은 단순하다 — **표 대신 신경망을 쓴다.**
상태를 넣으면 각 행동의 Q값을 내놓는 함수를 학습하는 것이다.

$$Q(s, a) \;\approx\; \text{신경망}(s)[a]$$

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import random
from collections import deque


class QNetwork(nn.Module):
    # 상태를 받아 각 행동의 Q값을 내놓는 신경망 (이론편 14.5절)

    def __init__(self, n_states=4, n_actions=2, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_states, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, n_actions),
        )

    def forward(self, x):
        return self.net(x)


class ReplayBuffer:
    # 경험 재생 버퍼 (이론편 14.5절)
    #
    # 왜 필요한가:
    #   연속된 경험은 서로 비슷해서, 그대로 학습하면 최근 상황에만 치우친다.
    #   버퍼에 모아 두고 무작위로 꺼내 쓰면 이 문제가 완화된다.

    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)

    def push(self, s, a, r, s2, done):
        self.buffer.append((s, a, r, s2, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        s, a, r, s2, d = zip(*batch)
        return (torch.tensor(np.array(s), dtype=torch.float32),
                torch.tensor(a, dtype=torch.int64),
                torch.tensor(r, dtype=torch.float32),
                torch.tensor(np.array(s2), dtype=torch.float32),
                torch.tensor(d, dtype=torch.float32))

    def __len__(self):
        return len(self.buffer)


print("=" * 55)
print("DQN 구성 요소")
print("=" * 55)
q = QNetwork()
print(f"신경망 파라미터: {sum(p.numel() for p in q.parameters()):,}개")
print()
print("입력 예시: CartPole 상태 4개 값")
sample_state = torch.tensor([[0.02, -0.01, 0.03, 0.02]])
with torch.no_grad():
    q_values = q(sample_state)
print(f"  상태 {sample_state.numpy().round(3)}")
print(f"  → Q값 {q_values.numpy().round(4)}   (왼쪽, 오른쪽)")
print(f"  → 선택할 행동: {q_values.argmax().item()}")
print()
print("표와 달리 '본 적 없는 상태'에도 답을 낼 수 있다 —")
print("이것이 신경망을 쓰는 이유다 (이론편 14.5절).")

In [ ]:
import gymnasium as gym
import torch
import torch.nn as nn
import numpy as np
import random
import time

def train_dqn(episodes=200, seed=42, verbose_every=40):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    env = gym.make("CartPole-v1")
    q_net = QNetwork()
    optimizer = torch.optim.Adam(q_net.parameters(), lr=1e-3)
    buffer = ReplayBuffer()

    GAMMA = 0.99
    BATCH = 64
    rewards_history = []

    t0 = time.time()
    for ep in range(episodes):
        s, _ = env.reset(seed=ep)
        eps = max(0.05, 1.0 - ep / (episodes * 0.5))
        total = 0.0

        for _ in range(500):
            # 행동 선택 (ε-greedy)
            if np.random.rand() < eps:
                a = env.action_space.sample()
            else:
                with torch.no_grad():
                    a = int(q_net(torch.tensor(s, dtype=torch.float32)).argmax())

            s2, r, term, trunc, _ = env.step(a)
            buffer.push(s, a, r, s2, float(term))
            s = s2
            total += r

            # 학습
            if len(buffer) >= BATCH:
                bs, ba, br, bs2, bd = buffer.sample(BATCH)
                # 목표값: r + gamma * max Q(s') (종료면 r만)
                with torch.no_grad():
                    target = br + GAMMA * q_net(bs2).max(1)[0] * (1 - bd)
                current = q_net(bs).gather(1, ba.unsqueeze(1)).squeeze(1)
                loss = nn.functional.mse_loss(current, target)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            if term or trunc:
                break

        rewards_history.append(total)
        if (ep + 1) % verbose_every == 0:
            recent = np.mean(rewards_history[-20:])
            print(f"  에피소드 {ep+1:4}: 최근 20회 평균 {recent:6.1f}점  "
                  f"(eps={eps:.2f}, {time.time()-t0:.0f}초)")

    env.close()
    return q_net, rewards_history


print("=" * 60)
print("DQN 학습 (CartPole, 200 에피소드)")
print("=" * 60)
dqn, dqn_rewards = train_dqn()
print("-" * 60)
print(f"초반 20회 평균: {np.mean(dqn_rewards[:20]):.1f}점")
print(f"후반 20회 평균: {np.mean(dqn_rewards[-20:]):.1f}점")
print(f"1절 무작위 행동: 약 20점")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4.5))

ax.plot(dqn_rewards, alpha=0.3, color="#94A3B8", linewidth=1, label="에피소드별")
window = 20
if len(dqn_rewards) >= window:
    smooth = np.convolve(dqn_rewards, np.ones(window)/window, mode="valid")
    ax.plot(range(window-1, len(dqn_rewards)), smooth,
            color="#EA580C", linewidth=2.5, label=f"{window}회 이동평균")

ax.axhline(np.mean(dqn_rewards[:20]), color="#64748B", linestyle=":",
           linewidth=1.5, label="학습 전 수준")
ax.set_xlabel("에피소드")
ax.set_ylabel("총 보상 (버틴 스텝 수)")
ax.set_title("DQN 학습 곡선")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("곡선이 들쭉날쭉한 것이 정상이다.")
print("지도학습과 달리 강화학습은 데이터를 스스로 만들어 내므로,")
print("정책이 조금만 바뀌어도 겪는 상황이 크게 달라진다.")

### 학습된 정책 확인

무작위 행동(1절)과 학습된 정책을 비교한다.

In [ ]:
import gymnasium as gym
import torch
import numpy as np

env = gym.make("CartPole-v1")

print("=" * 55)
print("학습 전후 비교")
print("=" * 55)

# 무작위
random_scores = []
for ep in range(20):
    s, _ = env.reset(seed=1000 + ep)
    total = 0
    for _ in range(500):
        s, r, term, trunc, _ = env.step(env.action_space.sample())
        total += r
        if term or trunc: break
    random_scores.append(total)

# 학습된 DQN (탐험 없이 최선만)
dqn_scores = []
for ep in range(20):
    s, _ = env.reset(seed=1000 + ep)
    total = 0
    for _ in range(500):
        with torch.no_grad():
            a = int(dqn(torch.tensor(s, dtype=torch.float32)).argmax())
        s, r, term, trunc, _ = env.step(a)
        total += r
        if term or trunc: break
    dqn_scores.append(total)

env.close()

print(f"{'방식':<16}{'평균':<12}{'최소':<10}{'최대'}")
print("-" * 55)
print(f"{'무작위':<16}{np.mean(random_scores):<12.1f}{min(random_scores):<10.0f}{max(random_scores):.0f}")
print(f"{'학습된 DQN':<16}{np.mean(dqn_scores):<12.1f}{min(dqn_scores):<10.0f}{max(dqn_scores):.0f}")
print("-" * 55)
print(f"개선 배수: {np.mean(dqn_scores)/np.mean(random_scores):.1f}배")
print()
print("정답을 한 번도 알려주지 않았는데도 막대를 세우는 법을 배웠다.")
print("보상 신호만으로 학습이 이루어진 것이다 (이론편 14.1절).")

---

## 7. 강화학습이 어려운 이유 — 이론편 14.6절

이론편 14.6절에서 강화학습의 실용적 어려움을 다뤘다. 코드를 돌려 보며 겪은 것들을 정리한다.

| 어려움 | 이 장에서 본 것 |
|---|---|
| 학습이 불안정 | 6절 곡선이 들쭉날쭉 |
| 시드에 민감 | 같은 코드도 시드 따라 결과가 다름 |
| 보상 설계가 어려움 | FrozenLake는 목표에서만 1점 (희소 보상) |
| 표본 효율이 낮음 | CartPole에 200 에피소드 = 수만 스텝 |

**시드에 따른 차이**를 직접 확인해 보자.

In [ ]:
import numpy as np

print("=" * 55)
print("시드에 따른 결과 차이 (이론편 14.6절)")
print("=" * 55)
print("같은 코드, 같은 설정, 시드만 다름")
print()

results = []
for seed in [0, 1, 2]:
    _, rews = train_dqn(episodes=100, seed=seed, verbose_every=1000)
    final = np.mean(rews[-20:])
    results.append(final)
    print(f"  시드 {seed}: 후반 20회 평균 {final:6.1f}점")

print("-" * 55)
print(f"최고 {max(results):.1f} / 최저 {min(results):.1f} / 차이 {max(results)-min(results):.1f}점")
print()
print("지도학습에서는 시드가 달라도 결과가 비슷하지만,")
print("강화학습은 초기 탐험이 어디로 향했느냐에 따라 크게 갈린다.")
print()
print("→ 논문에서 강화학습 결과를 보고할 때 여러 시드의 평균을 쓰는 이유다.")

---

## 8. 정리

### 확인한 이론편 값

| 이론편 절 | 내용 | 결과 |
|---|---|---|
| **14.3** | **Q(A) 0 → 2.25, Q(B) 0 → 5.0** | **일치** ✓ |
| 14.3 | 가치가 뒤에서 앞으로 전파 | 8칸 통로로 확인 ✓ |
| 14.4 | 탐험/활용 균형 | 세 전략 비교 ✓ |
| 14.5 | 표가 불가능한 연속 상태 | DQN으로 해결 ✓ |
| 14.6 | 시드에 민감 | 직접 확인 ✓ |

### 설치한 것

```
pip install gymnasium
```

예전 이름 `gym`이 아니라 **`gymnasium`**이며, `step()`이 5개 값을 돌려준다는 점이 다르다.

### 기억할 것

| 항목 | 요점 |
|---|---|
| 지도학습과의 차이 | 정답 없음, 보상만 있음 |
| Q-Learning | 목표값 − 현재 추정 = TD 오차만큼 갱신 |
| 가치 전파 | 보상 지점에서 앞으로 한 칸씩 |
| ε-greedy | 초기 탐험 많이 → 점차 활용 |
| DQN | 표 대신 신경망 — 연속 상태 처리 |
| 경험 재생 | 연속된 경험의 편향을 완화 |
| 재현성 | **여러 시드로 확인해야 함** |

### 다음 장

**18. Autoencoder와 VAE — 생성 모델의 시작** — 이론편 19장. 정답 없이 데이터의 구조를 배우는 또 다른 방식,
**생성 모델**로 넘어간다. 이론편 15.4절의 재파라미터화를 직접 구현한다.